# Confusion Matrix & Per-Class Analysis

In [1]:
import csv
import os
import random
import time
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
try:
    import timm
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "timm"])
    import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from skimage.filters import threshold_otsu
from sklearn.metrics import cohen_kappa_score
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore", category=UserWarning, module="timm.layers.interpolate")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def resolve_path(candidates, default):
    for c in candidates:
        if c and os.path.exists(c):
            return c
    return default

DATA_DIR = resolve_path([os.getenv("DATA_DIR", ""), "./data", "./dataset", "/kaggle/input/lulc-chennai", "../data"], "./data")
OUT_DIR = os.getenv("OUT_DIR", "./outputs")
os.makedirs(OUT_DIR, exist_ok=True)
NUM_CLASSES = 8
CKPT_NAME = "best_model_final_5.pth"
CVA_WATER_NDWI_THRESHOLD = 0.35
DISPLAY_CONFIDENCE_TAU = 0.60

CONFIG = {
    "img_size": 128,
    "in_chans": 9,
    "num_classes": NUM_CLASSES,
    "batch_size": 32,
    "num_workers": 4,
    "lr": 5.0e-5,
    "weight_decay": 6.358359e-4,
    "focal_gamma": 0.6136,
    "backbone_lr_ratio": 0.2719,
    "epochs": 160,
    "freeze_epochs": 15,
    "early_stop_patience": 20,
    "grad_clip_norm": 1.0,
    "label_smoothing": 0.1,
    "class_weight_cap": 5.0,
    "change_pos_weight_cap": 8.0,
    "change_bce_weight": 0.70,
    "change_dice_weight": 0.30,
    "change_task_weight": 1.0,
    "backbone": "swin_tiny_patch4_window7_224",
    "attn_heads": 8,
    "cross_attn_heads": 8,
    "num_layers": 3,
}

print(f"Device: {device}")

Device: cuda


## Data Loading

In [2]:
patches_2017 = np.load(os.path.join(DATA_DIR, "patches_2017.npy"))
patches_2019 = np.load(os.path.join(DATA_DIR, "patches_2019.npy"))
patches_2024 = np.load(os.path.join(DATA_DIR, "patches_2024.npy"))
patches_2025 = np.load(os.path.join(DATA_DIR, "patches_2025.npy"))
labels = np.load(os.path.join(DATA_DIR, "label_patches.npy"))

YEAR_ARRAYS = {
    2017: patches_2017,
    2019: patches_2019,
    2024: patches_2024,
    2025: patches_2025,
}
LABEL_FOR_YEAR = {year: labels for year in YEAR_ARRAYS}

n_patches = labels.shape[0]
rng = np.random.RandomState(SEED)
shuffled_idx = rng.permutation(n_patches)
n_train = int(n_patches * 0.80)
train_patch_idx = shuffled_idx[:n_train]
val_patch_idx = shuffled_idx[n_train:]
print(f"Split: {len(train_patch_idx)} train / {len(val_patch_idx)} val")


Split: 2774 train / 694 val


## Spectral Processing

In [3]:
def to_nine_band(patch_5band, eps=1e-6):
    b2, b3, b4, b8, b11 = patch_5band[:5]
    ndvi = (b8 - b4) / (b8 + b4 + eps)
    ndwi = (b3 - b8) / (b3 + b8 + eps)
    ndbi = (b11 - b8) / (b11 + b8 + eps)
    mndwi = (b3 - b11) / (b3 + b11 + eps)
    return np.stack([b2, b3, b4, b8, b11, ndvi, ndwi, ndbi, mndwi], axis=0).astype(np.float32)


def normalize_bands(bands):
    out = bands.copy()
    out[:5] = np.clip(out[:5], 0, 1)
    out[5:] = (np.clip(out[5:], -1, 1) + 1) / 2
    return out

## Model Architecture

In [4]:
class UNetDecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(out_ch+skip_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),         nn.BatchNorm2d(out_ch), nn.ReLU(True))
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class ASPP(nn.Module):
    def __init__(self, in_ch, out_ch, rates=[1,2,4]):
        super().__init__()
        self.stages = nn.ModuleList([nn.Sequential(
            nn.Conv2d(in_ch,out_ch,1,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(True))])
        for r in rates:
            self.stages.append(nn.Sequential(
                nn.Conv2d(in_ch,out_ch,3,padding=r,dilation=r,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(True)))
        self.pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),nn.Conv2d(in_ch,out_ch,1,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(True))
        self.project = nn.Sequential(
            nn.Conv2d(out_ch*(len(rates)+2),out_ch,1,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(True))
    def forward(self, x):
        feats = [s(x) for s in self.stages]
        p = F.interpolate(self.pool(x), size=x.shape[-2:], mode="bilinear", align_corners=False)
        feats.append(p)
        return self.project(torch.cat(feats, dim=1))

def widen_patch_embed(backbone, in_chans):
    proj = backbone.patch_embed.proj; old_w = proj.weight.data
    new_proj = nn.Conv2d(in_chans, proj.out_channels, proj.kernel_size, proj.stride, proj.padding,
                         bias=proj.bias is not None)
    with torch.no_grad():
        new_proj.weight[:,:3] = old_w
        if in_chans > 3:
            new_proj.weight[:,3:] = old_w.mean(1,keepdim=True).repeat(1,in_chans-3,1,1)
        if proj.bias is not None: new_proj.bias[:] = proj.bias.data
    backbone.patch_embed.proj = new_proj

def to_nchw(feat, chs):
    if feat.dim()==4 and feat.shape[1]!=feat.shape[-1] and feat.shape[-1] in chs:
        feat = feat.permute(0,3,1,2).contiguous()
    return feat

class BSALayer(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        self.n1 = nn.LayerNorm(dim); self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.n2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim,dim*4),nn.GELU(),nn.Linear(dim*4,dim))
    def forward(self, x):
        h=self.n1(x); x=x+self.attn(h,h,h,need_weights=False)[0]
        h=self.n2(x); return x+self.mlp(h)

class BottleneckSelfAttention(nn.Module):
    def __init__(self, dim, heads=8, num_layers=1):
        super().__init__()
        self.layers = nn.ModuleList([BSALayer(dim,heads) for _ in range(num_layers)])
        self.norm   = nn.LayerNorm(dim)
    def forward(self, x):
        B,C,H,W = x.shape
        t = x.permute(0,2,3,1).reshape(B,H*W,C)
        for l in self.layers: t=l(t)
        return self.norm(t).reshape(B,H,W,C).permute(0,3,1,2)

class CrossTransformerBlock(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        self.na=nn.LayerNorm(dim); self.nb=nn.LayerNorm(dim)
        self.attn12=nn.MultiheadAttention(dim,heads,batch_first=True)
        self.attn21=nn.MultiheadAttention(dim,heads,batch_first=True)
        self.nc=nn.LayerNorm(dim); self.nd=nn.LayerNorm(dim)
        self.proj=nn.Linear(dim*2,dim)
    def forward(self, f1, f2):
        B,C,H,W = f1.shape
        t1=f1.permute(0,2,3,1).reshape(B,H*W,C); t2=f2.permute(0,2,3,1).reshape(B,H*W,C)
        n1,n2=self.na(t1),self.nb(t2)
        c1=t1+self.attn12(n1,n2,n2,need_weights=False)[0]
        c2=t2+self.attn21(n2,n1,n1,need_weights=False)[0]
        out=self.proj(torch.cat([self.nc(c1),self.nd(c2)],dim=-1))
        return out.reshape(B,H,W,C).permute(0,3,1,2)

class SiameseSwinRefined(nn.Module):
    def __init__(self, num_classes=8, in_chans=9, img_size=128,
                 backbone_name="swin_tiny_patch4_window7_224",
                 pretrained=True, attn_heads=8, cross_heads=8, num_layers=3):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                            features_only=True, img_size=img_size, in_chans=3)
        widen_patch_embed(self.backbone, in_chans)
        self.chs = self.backbone.feature_info.channels()

        self.self_attn   = BottleneckSelfAttention(self.chs[3], attn_heads, num_layers)
        self.aspp        = ASPP(self.chs[3], self.chs[3], rates=[1,2,4])
        self.cross_attn  = CrossTransformerBlock(self.chs[3], cross_heads)

        self.dec3 = UNetDecoderBlock(self.chs[3], self.chs[2], self.chs[2])
        self.dec2 = UNetDecoderBlock(self.chs[2], self.chs[1], self.chs[1])
        self.dec1 = UNetDecoderBlock(self.chs[1], self.chs[0], self.chs[0])

        self.lulc_head = nn.Sequential(
            nn.Conv2d(self.chs[0], self.chs[0]//2, 3, padding=1),
            nn.BatchNorm2d(self.chs[0]//2), nn.ReLU(True),
            nn.Conv2d(self.chs[0]//2, num_classes, 1))

        self.change_head = nn.Sequential(
            nn.Conv2d(self.chs[3], self.chs[3]//2, 3, padding=1),
            nn.BatchNorm2d(self.chs[3]//2), nn.ReLU(True),
            nn.Conv2d(self.chs[3]//2, 1, 1))

        self.img_size = img_size

    def encode(self, x):
        return [to_nchw(f, self.chs) for f in self.backbone(x)]

    def forward(self, x1, x2):
        f1, f2 = self.encode(x1), self.encode(x2)
        f1a, f2a = self.self_attn(f1[3]), self.self_attn(f2[3])
        f1p, f2p = self.aspp(f1a), self.aspp(f2a)

        s1,s2,s3,_ = f2
        d3 = self.dec3(f2p, s3); d2 = self.dec2(d3,s2); d1 = self.dec1(d2,s1)
        lulc = F.interpolate(self.lulc_head(d1),(self.img_size,self.img_size),mode="bilinear",align_corners=False)

        fused = self.cross_attn(f1p, f2p)
        change = F.interpolate(self.change_head(fused),(self.img_size,self.img_size),mode="bilinear",align_corners=False)
        return lulc, change.squeeze(1)

## Load Checkpoint

In [5]:
model = SiameseSwinRefined(
    num_classes=NUM_CLASSES, in_chans=CONFIG["in_chans"], img_size=CONFIG["img_size"],
    backbone_name=CONFIG["backbone"], pretrained=False,
    attn_heads=CONFIG["attn_heads"], cross_heads=CONFIG["cross_attn_heads"],
    num_layers=CONFIG["num_layers"]
).to(device)

ckpt_path = os.path.join("/kaggle/input/datasets/user/lulc-model-1", CKPT_NAME)
checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")


Loaded checkpoint from epoch 158


## TTA

In [6]:
def tta_predict(model, x1, x2):
    lulc_sum = None
    change_sum = None
    for rotation in range(4):
        rotated_x1 = torch.rot90(x1, rotation, (2, 3))
        rotated_x2 = torch.rot90(x2, rotation, (2, 3))
        with torch.no_grad(), torch.amp.autocast("cuda", enabled=device.type == "cuda"):
            lulc_logits, change_logits = model(rotated_x1, rotated_x2)
        lulc_logits = torch.rot90(lulc_logits, -rotation, (2, 3))
        change_logits = torch.rot90(change_logits.unsqueeze(1), -rotation, (2, 3)).squeeze(1)
        lulc_sum = lulc_logits if lulc_sum is None else lulc_sum + lulc_logits
        change_sum = change_logits if change_sum is None else change_sum + change_logits
    return lulc_sum / 4, change_sum / 4

## Build Validation Loader

In [7]:
YEAR_PAIRS = [
    (2017, 2019), (2017, 2024), (2017, 2025),
    (2019, 2024), (2019, 2025), (2024, 2025),
]

class PairIndex:
    def __init__(self, pairs, patch_indices):
        self.items = [(y1, y2, i) for y1, y2 in pairs for i in patch_indices]
    def __len__(self):
        return len(self.items)

class SiameseLULCDataset(Dataset):
    def __init__(self, pair_index, patches_by_year, labels_by_year, augment=False):
        self.items = pair_index.items
        self.patches = patches_by_year
        self.labels = labels_by_year
        self.augment = augment

    def __len__(self):
        return len(self.items)

    def __getitem__(self, item_index):
        year1, year2, patch_index = self.items[item_index]
        img1 = self.patches[year1][patch_index].copy()
        img2 = self.patches[year2][patch_index].copy()
        label = self.labels[year2][patch_index].astype(np.int64)
        img1 = normalize_bands(to_nine_band(img1))
        img2 = normalize_bands(to_nine_band(img2))
        return torch.from_numpy(img1), torch.from_numpy(img2), torch.from_numpy(label)

val_index = PairIndex(YEAR_PAIRS, val_patch_idx)
val_loader = DataLoader(
    SiameseLULCDataset(val_index, YEAR_ARRAYS, LABEL_FOR_YEAR),
    batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"],
    pin_memory=True, persistent_workers=True,
)
print(f"Validation batches: {len(val_loader)}")

Validation batches: 131


## Generate Confusion Matrix

In [8]:
try:
    import seaborn as sns
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "seaborn"])
    import seaborn as sns

CLASS_NAMES = ["Trees", "Shrubland", "Grassland", "Cropland", "Built-up", "Bare/Sparse", "Water", "Wetland"]

all_preds = []
all_labels = []

model.eval()
print("Running TTA inference on validation set...")
for x1, x2, label in val_loader:
    x1 = x1.to(device, non_blocking=True)
    x2 = x2.to(device, non_blocking=True)
    lulc_logits, _ = tta_predict(model, x1, x2)
    preds = lulc_logits.argmax(1).cpu().numpy().reshape(-1)
    truth = label.numpy().reshape(-1)
    all_preds.append(preds)
    all_labels.append(truth)

all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)
print(f"Total pixels evaluated: {len(all_preds):,}")

Running TTA inference on validation set...
Total pixels evaluated: 68,222,976


## Confusion Matrix Heatmap

In [9]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_labels, all_preds, labels=range(NUM_CLASSES))
cm_normalized = cm.astype(np.float64) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(24, 10))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=axes[0], linewidths=0.5)
axes[0].set_title("Confusion Matrix (Raw Counts)", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Predicted Class", fontsize=12)
axes[0].set_ylabel("True Class", fontsize=12)

sns.heatmap(cm_normalized, annot=True, fmt=".3f", cmap="YlOrRd", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=axes[1], linewidths=0.5, vmin=0, vmax=1)
axes[1].set_title("Normalized Confusion Matrix (Row %)", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Predicted Class", fontsize=12)
axes[1].set_ylabel("True Class", fontsize=12)

fig.suptitle("8-Class LULC Classification — Confusion Matrix Analysis", fontsize=16, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"), dpi=200, bbox_inches="tight")
plt.show()
print("Saved confusion_matrix.png")

Saved confusion_matrix.png


## Per-Class Precision, Recall, F1, IoU

In [10]:
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(all_labels, all_preds, labels=range(NUM_CLASSES), zero_division=0)

iou = np.zeros(NUM_CLASSES)
for c in range(NUM_CLASSES):
    tp = cm[c, c]
    fp = cm[:, c].sum() - tp
    fn = cm[c, :].sum() - tp
    iou[c] = tp / (tp + fp + fn + 1e-12)

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
x = np.arange(NUM_CLASSES)
width = 0.6

for ax, values, title, color in [
    (axes[0,0], precision*100, "Precision (%)", "#2196F3"),
    (axes[0,1], recall*100, "Recall (%)", "#4CAF50"),
    (axes[1,0], f1*100, "F1-Score (%)", "#FF9800"),
    (axes[1,1], iou*100, "IoU (%)", "#9C27B0"),
]:
    bars = ax.bar(x, values, width, color=color, alpha=0.85, edgecolor="black", linewidth=0.5)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=10)
    ax.set_ylim(0, 105)
    ax.grid(axis="y", alpha=0.3)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f"{val:.1f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

fig.suptitle("Per-Class Performance Metrics (TTA)", fontsize=16, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "per_class_metrics.png"), dpi=200, bbox_inches="tight")
plt.show()
print("Saved per_class_metrics.png")

Saved per_class_metrics.png


## Top Misclassification Pairs

In [11]:
misclass_pairs = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i != j and cm[i, j] > 0:
            misclass_pairs.append((CLASS_NAMES[i], CLASS_NAMES[j], cm[i, j], cm_normalized[i, j]))

misclass_pairs.sort(key=lambda x: x[2], reverse=True)
print("Top-10 Misclassification Pairs:")
print(f"{'True Class':>15s} -> {'Predicted':>15s} | {'Count':>10s} | {'Rate':>8s}")
print("-" * 60)
for true_cls, pred_cls, count, rate in misclass_pairs[:10]:
    print(f"{true_cls:>15s} -> {pred_cls:>15s} | {count:>10,d} | {rate:>7.2%}")

Top-10 Misclassification Pairs:
     True Class ->       Predicted |      Count |     Rate
------------------------------------------------------------
          Trees ->        Built-up |  1,604,744 |  12.14%
       Built-up ->           Trees |  1,417,138 |   6.37%
       Cropland ->           Trees |    964,077 |   9.73%
          Trees ->        Cropland |    928,818 |   7.02%
       Cropland ->       Grassland |    775,058 |   7.82%
      Grassland ->        Cropland |    579,722 |  13.33%
       Built-up ->       Grassland |    482,729 |   2.17%
          Trees ->       Grassland |    375,686 |   2.84%
      Grassland ->        Built-up |    336,127 |   7.73%
      Grassland ->           Trees |    276,335 |   6.35%
